# Transformation - Combining flight data and Timezones

In [1]:
import sys
from pathlib import Path

project_root = "/notebooks/ML/Flight Delay Prediction Project/ELT"
sys.path.append(str(project_root))

In [2]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pandas as pd

In [3]:
from src.utils.spark_session import get_spark_session
from src.load.flight_data import *
from src.load.timezones import *

In [4]:
pd.set_option('display.max_columns', None)

In [ ]:
spark = get_spark_session()

In [ ]:
s3_key_fd = "cleaned/Flight_Delay_Prediction_Datasets/flight_data/"
s3_key_tz = "bronze/Flight_Delay_Prediction_Datasets/timezones/airport_timezones.parquet"

fd_df = get_flight_lake_dataset(spark, s3_key_fd)
tz_df = get_tz_lake_dataset(spark, s3_key_tz)

In [ ]:
fd_df.limit(5).toPandas()

In [ ]:
tz_df.limit(5).toPandas()

## Perform JOIN operation

In [9]:
from src.transform.combine_timezones import *

In [10]:
joined_df = combine_flights_timezones(spark, fd_df, tz_df)

## Validate data

In [ ]:
joined_df.limit(5).toPandas()

## Check for any missing values

In [12]:
import pyspark.sql.functions as F

In [13]:
tz_cols = ['origin_timezone', 'destination_timezone']

In [ ]:
joined_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in tz_cols
]).toPandas()

## Load data to data lake

In [15]:
s3_key = "cleaned/Flight_Delay_Prediction_Datasets/flight_data_with_tz/"
write_transformed_data(spark, joined_df, s3_key)